In [ ]:
# ============================================================
# DRWEIBO SSEE ABLATION MULTI-SEED EXPERIMENT
#
# Sparse Conversation Structure Modeling
# for Early Rumor Verification
#
# Controlled ablation study:
#   GAT / w/o SEM / w/o AGCC / w/o ASE / Full SSEE
#
# V2 change:
#   Add feature-space alignment before adaptive fusion:
#
#   semantic_rep  -> Linear + LayerNorm -> h_x_aligned
#   structural_rep-> Linear + LayerNorm -> h_e_aligned
#
#   h_f = lambda * h_e_aligned + (1-lambda) * h_x_aligned
#
# IMPORTANT:
#   GAT / SEM / AGCC / ASE / data split / seed / optimizer
#   protocol are unchanged from the previous full-model run.
# ============================================================

import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import gc
import json
import random
import time
from pathlib import Path
from collections import Counter, deque

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

from transformers import (
    BertTokenizer,
    BertModel,
    RobertaTokenizer,
    RobertaModel,
)

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

from tqdm import tqdm


# ============================================================
# 1. PATH CONFIG
# ============================================================

BASE_DIR = Path("/root/autodl-fs/processed_scsr")

SPLIT_DIR = BASE_DIR / "splits"

OUTPUT_DIR = (
    BASE_DIR /
    "experiment_results" /
    "full_model_v3"
)


# ============================================================
# 2. DATASET
# ============================================================

DATASET = "DRWeibo"

# Later:
# DATASET = "PHEME"

PHEME_FOLD = 1


# ============================================================
# 3. LOCAL PRETRAINED MODELS
# ============================================================

DRWEIBO_MODEL_DIR = (
    BASE_DIR /
    "chinese_roberta_wwm_ext"
)

PHEME_MODEL_DIR = (
    BASE_DIR /
    "roberta_base"
)


# ============================================================
# 4. REPRODUCIBILITY
# ============================================================

SEED = 42


# ============================================================
# 5. TEXT ENCODER
# ============================================================

MAX_LENGTH = 128
NODE_CHUNK_SIZE = 96


# ============================================================
# 6. BATCH
# ============================================================

BATCH_SIZE = 8
GRAD_ACCUM_STEPS = 1


# ============================================================
# 7. GAT
# ============================================================

GAT_HIDDEN_DIM = 256
GAT_HEADS = 4

assert GAT_HIDDEN_DIM % GAT_HEADS == 0

GAT_DROPOUT = 0.3
GAT_ATTENTION_DROPOUT = 0.2
LEAKY_RELU_SLOPE = 0.2

MAKE_BIDIRECTIONAL = True


# ============================================================
# 8. SSEE
# ============================================================

SEM_EVIDENCE_DIM = 128
GRAPH_STAT_DIM = 6
SSEE_DROPOUT = 0.3


# ============================================================
# 9. SAF
# ============================================================

SPARSITY_HIDDEN_DIM = 64
SAF_DROPOUT = 0.2

# Hidden dimension of the representation-aware fusion gate
FUSION_GATE_HIDDEN_DIM = 128

# Common fusion-space dimensionality.
FUSION_DIM = GAT_HIDDEN_DIM


# ============================================================
# 10. OPTIMIZATION
# ============================================================

ENCODER_LR = 2e-5
NEW_MODULE_LR = 1e-3

WEIGHT_DECAY = 1e-4
DROPOUT = 0.3
GRADIENT_CLIP = 1.0


# ============================================================
# 11. TRAINING
# ============================================================

MAX_EPOCHS = 10
PATIENCE = 3
NUM_WORKERS = 8


# ============================================================
# 12. RANDOM SEED
# ============================================================

def set_seed(seed):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    # RTX 4090 throughput settings.
    # Seeds remain fixed. TF32 is used consistently for all PHEME variants.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        try:
            torch.set_float32_matmul_precision("high")
        except Exception:
            pass


# ============================================================
# 13. DEVICE
# ============================================================

def get_device():

    if torch.cuda.is_available():

        device = torch.device("cuda")

        prop = torch.cuda.get_device_properties(0)

        total_memory = (
            prop.total_memory /
            1024 ** 3
        )

        print("\nCUDA available")
        print("GPU:", torch.cuda.get_device_name(0))
        print(f"Total VRAM: {total_memory:.2f} GB")

        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")

        return device

    print("\nCUDA unavailable. Using CPU.")

    return torch.device("cpu")


# ============================================================
# 14. CUDA MEMORY
# ============================================================

def print_cuda_memory():

    if not torch.cuda.is_available():
        return

    allocated = (
        torch.cuda.memory_allocated() /
        1024 ** 3
    )

    reserved = (
        torch.cuda.memory_reserved() /
        1024 ** 3
    )

    peak = (
        torch.cuda.max_memory_allocated() /
        1024 ** 3
    )

    print(
        f"CUDA memory | "
        f"allocated={allocated:.2f} GB | "
        f"reserved={reserved:.2f} GB | "
        f"peak={peak:.2f} GB"
    )


# ============================================================
# 15. LOAD JSONL
# ============================================================

def load_jsonl(path):

    samples = []

    with open(path, "r", encoding="utf-8") as f:

        for line in f:

            line = line.strip()

            if not line:
                continue

            samples.append(
                json.loads(line)
            )

    return samples


# ============================================================
# 16. LABEL MAPPING
# ============================================================

def get_label_mapping(dataset):

    if dataset == "DRWeibo":

        return (
            {
                "0": 0,
                "1": 1,
            },
            {
                0: "0",
                1: "1",
            },
        )

    if dataset == "PHEME":

        return (
            {
                "false": 0,
                "true": 1,
                "unverified": 2,
            },
            {
                0: "false",
                1: "true",
                2: "unverified",
            },
        )

    raise ValueError(
        f"Unsupported dataset: {dataset}"
    )


# ============================================================
# 17. PATHS
# ============================================================

def get_paths():

    if DATASET == "DRWeibo":

        data_dir = SPLIT_DIR / "DRWeibo"
        model_dir = DRWEIBO_MODEL_DIR

    elif DATASET == "PHEME":

        data_dir = (
            SPLIT_DIR /
            "PHEME" /
            f"fold_{PHEME_FOLD}"
        )

        model_dir = PHEME_MODEL_DIR

    else:

        raise ValueError(DATASET)

    return (
        data_dir / "train.jsonl",
        data_dir / "val.jsonl",
        data_dir / "test.jsonl",
        model_dir,
    )


# ============================================================
# 18. TOKENIZER
# ============================================================

def load_tokenizer(dataset, model_dir):

    if dataset == "DRWeibo":

        return BertTokenizer.from_pretrained(
            str(model_dir),
            local_files_only=True
        )

    if dataset == "PHEME":

        return RobertaTokenizer.from_pretrained(
            str(model_dir),
            local_files_only=True
        )

    raise ValueError(dataset)


# ============================================================
# 19. GRAPH DEPTH
# ============================================================

def compute_max_depth(
    nodes,
    edges,
    root_id
):

    node_ids = {
        str(node["node_id"])
        for node in nodes
    }

    children = {
        node_id: []
        for node_id in node_ids
    }

    for edge in edges:

        if (
            not isinstance(edge, (list, tuple))
            or
            len(edge) != 2
        ):
            continue

        parent = str(edge[0])
        child = str(edge[1])

        if (
            parent in node_ids
            and
            child in node_ids
        ):
            children[parent].append(child)

    root_id = str(root_id)

    if root_id not in node_ids:
        return 0

    queue = deque(
        [(root_id, 0)]
    )

    visited = set()
    max_depth = 0

    while queue:

        node_id, depth = queue.popleft()

        if node_id in visited:
            continue

        visited.add(node_id)

        max_depth = max(
            max_depth,
            depth
        )

        for child in children.get(
            node_id,
            []
        ):
            queue.append(
                (child, depth + 1)
            )

    return max_depth


# ============================================================
# 20. GRAPH STATISTICS / SSD INPUT
# ============================================================

def extract_graph_statistics(sample):

    nodes = sample.get(
        "nodes",
        []
    )

    edges = sample.get(
        "edges",
        []
    )

    n = len(nodes)
    e = len(edges)

    existing = sample.get(
        "graph_statistics",
        {}
    )

    depth = existing.get(
        "max_depth",
        existing.get(
            "depth",
            None
        )
    )

    if depth is None:

        root_id = sample.get(
            "root_id",
            (
                nodes[0]["node_id"]
                if nodes
                else ""
            )
        )

        depth = compute_max_depth(
            nodes,
            edges,
            root_id
        )

    # Directed graph density.
    if n > 1:
        density = (
            e /
            (
                n *
                (n - 1)
            )
        )
    else:
        density = 0.0

    # Average degree.
    if n > 0:
        avg_degree = (
            2.0 *
            e /
            n
        )
    else:
        avg_degree = 0.0

    # Same working definition used in the previous SSEE/full run.
    branching = (
        e /
        max(
            float(depth),
            1.0
        )
    )

    # Fixed numerical scaling.
    stats = np.array(
        [
            np.log1p(n),
            np.log1p(e),
            np.log1p(float(depth)),
            float(density),
            np.log1p(avg_degree),
            np.log1p(branching),
        ],
        dtype=np.float32
    )

    return stats


# ============================================================
# 21. DATASET
# ============================================================

class ConversationGraphDataset(
    Dataset
):

    def __init__(
        self,
        samples,
        label2id
    ):

        self.samples = samples
        self.label2id = label2id


    def __len__(self):

        return len(
            self.samples
        )


    def __getitem__(
        self,
        idx
    ):

        sample = self.samples[idx]

        nodes = sample.get(
            "nodes",
            []
        )

        edges = sample.get(
            "edges",
            []
        )

        node_id_to_idx = {}
        texts = []

        for node_idx, node in enumerate(
            nodes
        ):

            node_id = str(
                node["node_id"]
            )

            node_id_to_idx[
                node_id
            ] = node_idx

            text = str(
                node.get(
                    "text",
                    ""
                )
            ).strip()

            if not text:
                text = "[EMPTY]"

            texts.append(text)

        indexed_edges = []

        for edge in edges:

            if (
                not isinstance(edge, (list, tuple))
                or
                len(edge) != 2
            ):
                continue

            parent = str(edge[0])
            child = str(edge[1])

            if (
                parent in node_id_to_idx
                and
                child in node_id_to_idx
            ):

                indexed_edges.append(
                    (
                        node_id_to_idx[parent],
                        node_id_to_idx[child],
                    )
                )

        label = self.label2id[
            str(
                sample["label"]
            )
        ]

        graph_stats = (
            extract_graph_statistics(
                sample
            )
        )

        return {
            "id": str(sample["id"]),
            "texts": texts,
            "edges": indexed_edges,
            "label": label,
            "graph_stats": graph_stats,
        }


# ============================================================
# 22. COLLATOR
# ============================================================

class GraphConversationCollator:

    def __init__(
        self,
        tokenizer,
        max_length
    ):

        self.tokenizer = tokenizer
        self.max_length = max_length


    def __call__(
        self,
        batch
    ):

        all_texts = []
        all_edges = []
        conversation_ids = []
        labels = []
        sample_ids = []
        graph_stats = []

        node_offset = 0

        for conv_idx, item in enumerate(
            batch
        ):

            texts = item["texts"]
            edges = item["edges"]

            num_nodes = len(texts)

            if num_nodes == 0:

                raise ValueError(
                    f"Conversation "
                    f"{item['id']} "
                    f"contains zero nodes."
                )

            all_texts.extend(texts)

            conversation_ids.extend(
                [conv_idx] *
                num_nodes
            )

            for src, dst in edges:

                all_edges.append(
                    (
                        src + node_offset,
                        dst + node_offset,
                    )
                )

            node_offset += num_nodes

            labels.append(
                item["label"]
            )

            sample_ids.append(
                item["id"]
            )

            graph_stats.append(
                item["graph_stats"]
            )

        encoded = self.tokenizer(
            all_texts,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        if all_edges:

            edge_index = (
                torch.tensor(
                    all_edges,
                    dtype=torch.long
                )
                .t()
                .contiguous()
            )

        else:

            edge_index = torch.empty(
                (2, 0),
                dtype=torch.long
            )

        return {
            "input_ids":
                encoded["input_ids"],

            "attention_mask":
                encoded["attention_mask"],

            "edge_index":
                edge_index,

            "conversation_ids":
                torch.tensor(
                    conversation_ids,
                    dtype=torch.long
                ),

            "graph_stats":
                torch.tensor(
                    np.stack(
                        graph_stats
                    ),
                    dtype=torch.float32
                ),

            "labels":
                torch.tensor(
                    labels,
                    dtype=torch.long
                ),

            "sample_ids":
                sample_ids,
        }


# ============================================================
# 23. GRAPH EDGE PREPARATION
# ============================================================

def prepare_edge_index(
    edge_index,
    num_nodes,
    make_bidirectional=True
):

    edges = edge_index
    device = edge_index.device

    if (
        make_bidirectional
        and
        edges.size(1) > 0
    ):

        reverse = torch.stack(
            [
                edges[1],
                edges[0],
            ],
            dim=0
        )

        edges = torch.cat(
            [
                edges,
                reverse,
            ],
            dim=1
        )

    node_idx = torch.arange(
        num_nodes,
        dtype=torch.long,
        device=device
    )

    self_loops = torch.stack(
        [
            node_idx,
            node_idx,
        ],
        dim=0
    )

    edges = torch.cat(
        [
            edges,
            self_loops,
        ],
        dim=1
    )

    edge_pairs = (
        edges
        .t()
        .contiguous()
    )

    edge_pairs = torch.unique(
        edge_pairs,
        dim=0
    )

    return (
        edge_pairs
        .t()
        .contiguous()
    )


# ============================================================
# 24. MULTI-HEAD EDGE SOFTMAX
# ============================================================

def edge_softmax(
    scores,
    dst,
    num_nodes
):

    scores_fp32 = scores.float()

    num_heads = (
        scores_fp32.size(1)
    )

    expanded_dst = (
        dst
        .unsqueeze(1)
        .expand(
            -1,
            num_heads
        )
    )

    max_per_node = torch.full(
        (
            num_nodes,
            num_heads
        ),
        -float("inf"),
        dtype=torch.float32,
        device=scores.device
    )

    max_per_node.scatter_reduce_(
        0,
        expanded_dst,
        scores_fp32,
        reduce="amax",
        include_self=True
    )

    stabilized = (
        scores_fp32
        -
        max_per_node[dst]
    )

    exp_scores = torch.exp(
        stabilized
    )

    denominator = torch.zeros(
        (
            num_nodes,
            num_heads
        ),
        dtype=torch.float32,
        device=scores.device
    )

    denominator.index_add_(
        0,
        dst,
        exp_scores
    )

    return (
        exp_scores /
        (
            denominator[dst]
            +
            1e-12
        )
    )


# ============================================================
# 25. SCALAR EDGE SOFTMAX
# ============================================================

def scalar_edge_softmax(
    scores,
    dst,
    num_nodes
):

    scores_fp32 = scores.float()

    max_per_node = torch.full(
        (num_nodes,),
        -float("inf"),
        dtype=torch.float32,
        device=scores.device
    )

    max_per_node.scatter_reduce_(
        0,
        dst,
        scores_fp32,
        reduce="amax",
        include_self=True
    )

    stabilized = (
        scores_fp32
        -
        max_per_node[dst]
    )

    exp_scores = torch.exp(
        stabilized
    )

    denominator = torch.zeros(
        (num_nodes,),
        dtype=torch.float32,
        device=scores.device
    )

    denominator.index_add_(
        0,
        dst,
        exp_scores
    )

    return (
        exp_scores /
        (
            denominator[dst]
            +
            1e-12
        )
    )


# ============================================================
# 26. VANILLA GAT LAYER
# ============================================================

class VanillaGATLayer(
    nn.Module
):

    def __init__(
        self,
        input_dim,
        output_dim,
        num_heads,
        dropout,
        attention_dropout,
        negative_slope=0.2
    ):

        super().__init__()

        assert (
            output_dim %
            num_heads ==
            0
        )

        self.output_dim = output_dim
        self.num_heads = num_heads

        self.head_dim = (
            output_dim //
            num_heads
        )

        self.negative_slope = negative_slope

        self.linear = nn.Linear(
            input_dim,
            output_dim,
            bias=False
        )

        self.att_src = nn.Parameter(
            torch.empty(
                num_heads,
                self.head_dim
            )
        )

        self.att_dst = nn.Parameter(
            torch.empty(
                num_heads,
                self.head_dim
            )
        )

        self.bias = nn.Parameter(
            torch.zeros(
                output_dim
            )
        )

        self.feature_dropout = nn.Dropout(
            dropout
        )

        self.attention_dropout = nn.Dropout(
            attention_dropout
        )

        self.reset_parameters()


    def reset_parameters(self):

        nn.init.xavier_uniform_(
            self.linear.weight
        )

        nn.init.xavier_uniform_(
            self.att_src
        )

        nn.init.xavier_uniform_(
            self.att_dst
        )

        nn.init.zeros_(
            self.bias
        )


    def forward(
        self,
        x,
        edge_index
    ):

        num_nodes = x.size(0)

        x = self.feature_dropout(x)

        h = self.linear(x)

        h = h.view(
            num_nodes,
            self.num_heads,
            self.head_dim
        )

        src = edge_index[0]
        dst = edge_index[1]

        src_score = (
            (
                h[src]
                *
                self.att_src
            )
            .sum(dim=-1)
        )

        dst_score = (
            (
                h[dst]
                *
                self.att_dst
            )
            .sum(dim=-1)
        )

        scores = F.leaky_relu(
            src_score + dst_score,
            negative_slope=
                self.negative_slope
        )

        alpha = edge_softmax(
            scores,
            dst,
            num_nodes
        )

        alpha = self.attention_dropout(
            alpha
        )

        # AMP dtype alignment
        alpha = alpha.to(
            dtype=h.dtype
        )

        messages = (
            h[src]
            *
            alpha.unsqueeze(-1)
        )

        output = torch.zeros(
            (
                num_nodes,
                self.num_heads,
                self.head_dim
            ),
            dtype=h.dtype,
            device=h.device
        )

        output.index_add_(
            0,
            dst,
            messages
        )

        output = output.reshape(
            num_nodes,
            self.output_dim
        )

        return (
            output +
            self.bias
        )


# ============================================================
# 27. SEM
# ============================================================

class StructuralEvidenceMining(
    nn.Module
):

    def __init__(
        self,
        hidden_dim,
        evidence_dim,
        dropout
    ):

        super().__init__()

        self.query_proj = nn.Linear(
            hidden_dim,
            evidence_dim,
            bias=False
        )

        self.key_proj = nn.Linear(
            hidden_dim,
            evidence_dim,
            bias=False
        )

        self.evidence_vector = nn.Parameter(
            torch.empty(
                evidence_dim
            )
        )

        self.dropout = nn.Dropout(
            dropout
        )

        self.reset_parameters()


    def reset_parameters(self):

        nn.init.xavier_uniform_(
            self.query_proj.weight
        )

        nn.init.xavier_uniform_(
            self.key_proj.weight
        )

        nn.init.normal_(
            self.evidence_vector,
            mean=0.0,
            std=0.02
        )


    def forward(
        self,
        h_g,
        edge_index
    ):

        num_nodes = h_g.size(0)

        src = edge_index[0]
        dst = edge_index[1]

        q_i = self.query_proj(
            h_g[dst]
        )

        k_j = self.key_proj(
            h_g[src]
        )

        evidence_hidden = torch.tanh(
            q_i + k_j
        )

        evidence_scores = (
            evidence_hidden
            *
            self.evidence_vector
        ).sum(dim=-1)

        alpha = scalar_edge_softmax(
            evidence_scores,
            dst,
            num_nodes
        )

        alpha = self.dropout(
            alpha
        )

        alpha = alpha.to(
            dtype=h_g.dtype
        )

        messages = (
            h_g[src]
            *
            alpha.unsqueeze(-1)
        )

        h_m = torch.zeros(
            h_g.shape,
            dtype=h_g.dtype,
            device=h_g.device
        )

        h_m.index_add_(
            0,
            dst,
            messages
        )

        return h_m


# ============================================================
# 28. AGCC
# ============================================================

class AdaptiveGlobalContextCompensation(
    nn.Module
):

    def __init__(
        self,
        hidden_dim,
        stat_dim
    ):

        super().__init__()

        self.gate = nn.Linear(
            hidden_dim * 2 +
            stat_dim,
            hidden_dim
        )


    def forward(
        self,
        h_m,
        conversation_ids,
        graph_stats,
        batch_size
    ):

        hidden_dim = h_m.size(-1)

        global_context = torch.zeros(
            (
                batch_size,
                hidden_dim
            ),
            dtype=h_m.dtype,
            device=h_m.device
        )

        global_context.index_add_(
            0,
            conversation_ids,
            h_m
        )

        counts = (
            torch.bincount(
                conversation_ids,
                minlength=batch_size
            )
            .clamp(min=1)
            .unsqueeze(1)
            .to(
                dtype=h_m.dtype,
                device=h_m.device
            )
        )

        global_context = (
            global_context /
            counts
        )

        node_global = (
            global_context[
                conversation_ids
            ]
        )

        node_stats = (
            graph_stats[
                conversation_ids
            ]
            .to(
                dtype=h_m.dtype
            )
        )

        gate_input = torch.cat(
            [
                h_m,
                node_global,
                node_stats,
            ],
            dim=-1
        )

        gamma = torch.sigmoid(
            self.gate(
                gate_input
            )
        )

        h_c = (
            h_m +
            gamma *
            node_global
        )

        return h_c


# ============================================================
# 29. ASE
# ============================================================

class AdaptiveStructuralEnhancement(
    nn.Module
):

    def __init__(
        self,
        hidden_dim,
        stat_dim,
        dropout
    ):

        super().__init__()

        self.mlp = nn.Sequential(
            nn.Linear(
                hidden_dim +
                stat_dim,
                hidden_dim
            ),
            nn.GELU(),
            nn.Dropout(
                dropout
            ),
            nn.Linear(
                hidden_dim,
                hidden_dim
            ),
        )


    def forward(
        self,
        h_c,
        h_g,
        conversation_ids,
        graph_stats
    ):

        node_stats = (
            graph_stats[
                conversation_ids
            ]
            .to(
                dtype=h_c.dtype
            )
        )

        ase_input = torch.cat(
            [
                h_c,
                node_stats,
            ],
            dim=-1
        )

        correction = self.mlp(
            ase_input
        )

        # Residual to initial GAT structural representation.
        h_e = (
            h_g +
            correction
        )

        return h_e



# ============================================================
# 30. CONTROLLED MODEL VARIANTS
# ============================================================

class SemanticOnlyClassifier(nn.Module):

    def __init__(self, dataset, model_dir, num_classes):
        super().__init__()

        if dataset == "DRWeibo":
            self.encoder = BertModel.from_pretrained(
                str(model_dir),
                local_files_only=True
            )
        elif dataset == "PHEME":
            self.encoder = RobertaModel.from_pretrained(
                str(model_dir),
                local_files_only=True
            )
        else:
            raise ValueError(dataset)

        self.encoder.gradient_checkpointing_enable()

        if hasattr(self.encoder.config, "use_cache"):
            self.encoder.config.use_cache = False

        self.dropout = nn.Dropout(DROPOUT)
        self.classifier = nn.Linear(
            self.encoder.config.hidden_size,
            num_classes
        )

    def encode_nodes(self, input_ids, attention_mask):
        chunks = []
        total_nodes = input_ids.size(0)

        for start in range(0, total_nodes, NODE_CHUNK_SIZE):
            end = min(start + NODE_CHUNK_SIZE, total_nodes)

            outputs = self.encoder(
                input_ids=input_ids[start:end],
                attention_mask=attention_mask[start:end]
            )

            chunks.append(
                outputs.last_hidden_state[:, 0, :]
            )

        return torch.cat(chunks, dim=0)

    @staticmethod
    def conversation_mean_pool(
        node_features,
        conversation_ids,
        batch_size
    ):
        hidden_dim = node_features.size(-1)

        pooled = torch.zeros(
            (batch_size, hidden_dim),
            dtype=node_features.dtype,
            device=node_features.device
        )

        pooled.index_add_(
            0,
            conversation_ids,
            node_features
        )

        counts = (
            torch.bincount(
                conversation_ids,
                minlength=batch_size
            )
            .clamp(min=1)
            .unsqueeze(1)
            .to(
                dtype=node_features.dtype,
                device=node_features.device
            )
        )

        return pooled / counts

    def forward(
        self,
        input_ids,
        attention_mask,
        edge_index,
        conversation_ids,
        graph_stats,
        batch_size
    ):
        h_x = self.encode_nodes(
            input_ids,
            attention_mask
        )

        h_graph = self.conversation_mean_pool(
            h_x,
            conversation_ids,
            batch_size
        )

        return self.classifier(
            self.dropout(h_graph)
        )


class GATOnlyClassifier(nn.Module):

    def __init__(self, dataset, model_dir, num_classes):
        super().__init__()

        if dataset == "DRWeibo":
            self.encoder = BertModel.from_pretrained(
                str(model_dir),
                local_files_only=True
            )
        elif dataset == "PHEME":
            self.encoder = RobertaModel.from_pretrained(
                str(model_dir),
                local_files_only=True
            )
        else:
            raise ValueError(dataset)

        self.encoder.gradient_checkpointing_enable()

        if hasattr(self.encoder.config, "use_cache"):
            self.encoder.config.use_cache = False

        encoder_dim = self.encoder.config.hidden_size

        self.gat1 = VanillaGATLayer(
            input_dim=encoder_dim,
            output_dim=GAT_HIDDEN_DIM,
            num_heads=GAT_HEADS,
            dropout=GAT_DROPOUT,
            attention_dropout=GAT_ATTENTION_DROPOUT,
            negative_slope=LEAKY_RELU_SLOPE
        )

        self.gat2 = VanillaGATLayer(
            input_dim=GAT_HIDDEN_DIM,
            output_dim=GAT_HIDDEN_DIM,
            num_heads=GAT_HEADS,
            dropout=GAT_DROPOUT,
            attention_dropout=GAT_ATTENTION_DROPOUT,
            negative_slope=LEAKY_RELU_SLOPE
        )

        self.dropout = nn.Dropout(DROPOUT)

        self.classifier = nn.Linear(
            GAT_HIDDEN_DIM,
            num_classes
        )

    def encode_nodes(self, input_ids, attention_mask):
        chunks = []
        total_nodes = input_ids.size(0)

        for start in range(0, total_nodes, NODE_CHUNK_SIZE):
            end = min(start + NODE_CHUNK_SIZE, total_nodes)

            outputs = self.encoder(
                input_ids=input_ids[start:end],
                attention_mask=attention_mask[start:end]
            )

            chunks.append(
                outputs.last_hidden_state[:, 0, :]
            )

        return torch.cat(chunks, dim=0)

    @staticmethod
    def conversation_mean_pool(
        node_features,
        conversation_ids,
        batch_size
    ):
        hidden_dim = node_features.size(-1)

        pooled = torch.zeros(
            (batch_size, hidden_dim),
            dtype=node_features.dtype,
            device=node_features.device
        )

        pooled.index_add_(
            0,
            conversation_ids,
            node_features
        )

        counts = (
            torch.bincount(
                conversation_ids,
                minlength=batch_size
            )
            .clamp(min=1)
            .unsqueeze(1)
            .to(
                dtype=node_features.dtype,
                device=node_features.device
            )
        )

        return pooled / counts

    def forward(
        self,
        input_ids,
        attention_mask,
        edge_index,
        conversation_ids,
        graph_stats,
        batch_size
    ):
        h_x = self.encode_nodes(
            input_ids,
            attention_mask
        )

        num_nodes = h_x.size(0)

        graph_edge_index = prepare_edge_index(
            edge_index,
            num_nodes,
            MAKE_BIDIRECTIONAL
        )

        h_g = F.elu(
            self.gat1(
                h_x,
                graph_edge_index
            )
        )

        h_g = F.elu(
            self.gat2(
                h_g,
                graph_edge_index
            )
        )

        h_graph = self.conversation_mean_pool(
            h_g,
            conversation_ids,
            batch_size
        )

        return self.classifier(
            self.dropout(h_graph)
        )


class GATSSEEClassifier(nn.Module):

    def __init__(self, dataset, model_dir, num_classes):
        super().__init__()

        if dataset == "DRWeibo":
            self.encoder = BertModel.from_pretrained(
                str(model_dir),
                local_files_only=True
            )
        elif dataset == "PHEME":
            self.encoder = RobertaModel.from_pretrained(
                str(model_dir),
                local_files_only=True
            )
        else:
            raise ValueError(dataset)

        self.encoder.gradient_checkpointing_enable()

        if hasattr(self.encoder.config, "use_cache"):
            self.encoder.config.use_cache = False

        encoder_dim = self.encoder.config.hidden_size

        self.gat1 = VanillaGATLayer(
            input_dim=encoder_dim,
            output_dim=GAT_HIDDEN_DIM,
            num_heads=GAT_HEADS,
            dropout=GAT_DROPOUT,
            attention_dropout=GAT_ATTENTION_DROPOUT,
            negative_slope=LEAKY_RELU_SLOPE
        )

        self.gat2 = VanillaGATLayer(
            input_dim=GAT_HIDDEN_DIM,
            output_dim=GAT_HIDDEN_DIM,
            num_heads=GAT_HEADS,
            dropout=GAT_DROPOUT,
            attention_dropout=GAT_ATTENTION_DROPOUT,
            negative_slope=LEAKY_RELU_SLOPE
        )

        self.sem = StructuralEvidenceMining(
            hidden_dim=GAT_HIDDEN_DIM,
            evidence_dim=SEM_EVIDENCE_DIM,
            dropout=SSEE_DROPOUT
        )

        self.agcc = AdaptiveGlobalContextCompensation(
            hidden_dim=GAT_HIDDEN_DIM,
            stat_dim=GRAPH_STAT_DIM
        )

        self.ase = AdaptiveStructuralEnhancement(
            hidden_dim=GAT_HIDDEN_DIM,
            stat_dim=GRAPH_STAT_DIM,
            dropout=SSEE_DROPOUT
        )

        self.dropout = nn.Dropout(DROPOUT)

        self.classifier = nn.Linear(
            GAT_HIDDEN_DIM,
            num_classes
        )

    def encode_nodes(self, input_ids, attention_mask):
        chunks = []
        total_nodes = input_ids.size(0)

        for start in range(0, total_nodes, NODE_CHUNK_SIZE):
            end = min(start + NODE_CHUNK_SIZE, total_nodes)

            outputs = self.encoder(
                input_ids=input_ids[start:end],
                attention_mask=attention_mask[start:end]
            )

            chunks.append(
                outputs.last_hidden_state[:, 0, :]
            )

        return torch.cat(chunks, dim=0)

    @staticmethod
    def conversation_mean_pool(
        node_features,
        conversation_ids,
        batch_size
    ):
        hidden_dim = node_features.size(-1)

        pooled = torch.zeros(
            (batch_size, hidden_dim),
            dtype=node_features.dtype,
            device=node_features.device
        )

        pooled.index_add_(
            0,
            conversation_ids,
            node_features
        )

        counts = (
            torch.bincount(
                conversation_ids,
                minlength=batch_size
            )
            .clamp(min=1)
            .unsqueeze(1)
            .to(
                dtype=node_features.dtype,
                device=node_features.device
            )
        )

        return pooled / counts

    def forward(
        self,
        input_ids,
        attention_mask,
        edge_index,
        conversation_ids,
        graph_stats,
        batch_size
    ):
        h_x = self.encode_nodes(
            input_ids,
            attention_mask
        )

        num_nodes = h_x.size(0)

        graph_edge_index = prepare_edge_index(
            edge_index,
            num_nodes,
            MAKE_BIDIRECTIONAL
        )

        h_g = F.elu(
            self.gat1(
                h_x,
                graph_edge_index
            )
        )

        h_g = F.elu(
            self.gat2(
                h_g,
                graph_edge_index
            )
        )

        h_m = self.sem(
            h_g,
            graph_edge_index
        )

        h_c = self.agcc(
            h_m,
            conversation_ids,
            graph_stats,
            batch_size
        )

        h_e = self.ase(
            h_c,
            h_g,
            conversation_ids,
            graph_stats
        )

        h_graph = self.conversation_mean_pool(
            h_e,
            conversation_ids,
            batch_size
        )

        return self.classifier(
            self.dropout(h_graph)
        )


# ============================================================
# 30B. SSEE REMOVE-ONE ABLATION MODEL
# ============================================================

class SSEEAblationClassifier(GATSSEEClassifier):

    """
    Controlled remove-one ablations of SSEE.

    full SSEE:
        H^G -> SEM -> AGCC -> ASE -> pooling

    wo_sem:
        H^G --------> AGCC -> ASE -> pooling

    wo_agcc:
        H^G -> SEM --------> ASE -> pooling

    wo_ase:
        H^G -> SEM -> AGCC -----> pooling

    No new module is introduced in any ablation.
    """

    def __init__(
        self,
        dataset,
        model_dir,
        num_classes,
        ablation
    ):

        super().__init__(
            dataset,
            model_dir,
            num_classes
        )

        valid = {
            "wo_sem",
            "wo_agcc",
            "wo_ase",
        }

        if ablation not in valid:
            raise ValueError(
                f"Unsupported ablation: {ablation}"
            )

        self.ablation = ablation


    def forward(
        self,
        input_ids,
        attention_mask,
        edge_index,
        conversation_ids,
        graph_stats,
        batch_size
    ):

        h_x = self.encode_nodes(
            input_ids,
            attention_mask
        )

        num_nodes = h_x.size(0)

        graph_edge_index = prepare_edge_index(
            edge_index,
            num_nodes,
            MAKE_BIDIRECTIONAL
        )

        h_g = F.elu(
            self.gat1(
                h_x,
                graph_edge_index
            )
        )

        h_g = F.elu(
            self.gat2(
                h_g,
                graph_edge_index
            )
        )

        # ----------------------------------------------------
        # SEM
        # ----------------------------------------------------
        if self.ablation == "wo_sem":
            h_m = h_g
        else:
            h_m = self.sem(
                h_g,
                graph_edge_index
            )

        # ----------------------------------------------------
        # AGCC
        # ----------------------------------------------------
        if self.ablation == "wo_agcc":
            h_c = h_m
        else:
            h_c = self.agcc(
                h_m,
                conversation_ids,
                graph_stats,
                batch_size
            )

        # ----------------------------------------------------
        # ASE
        # ----------------------------------------------------
        if self.ablation == "wo_ase":
            h_e = h_c
        else:
            h_e = self.ase(
                h_c,
                h_g,
                conversation_ids,
                graph_stats
            )

        h_graph = self.conversation_mean_pool(
            h_e,
            conversation_ids,
            batch_size
        )

        return self.classifier(
            self.dropout(h_graph)
        )


# ============================================================
# 31. EARLY-WINDOW EXPERIMENT CONFIG
# ============================================================

WINDOWS = [10, 30, 60, 120, 240]

MODEL_VARIANTS = [
    "semantic",
    "gat",
    "ssee",
]

# To test SSEE only first:
# MODEL_VARIANTS = ["ssee"]

EARLY_DATA_DIR = (
    BASE_DIR /
    "DRWeibo"
)

EARLY_RESULTS_DIR = (
    BASE_DIR /
    "experiment_results" /
    "drweibo_early_windows"
)


# ============================================================
# 32. EARLY-WINDOW FILE
# ============================================================

def get_early_window_path(window):

    candidates = [
        EARLY_DATA_DIR /
        f"drweibo_{window}min.jsonl",

        BASE_DIR /
        f"drweibo_{window}min.jsonl",

        BASE_DIR /
        "DRWeibo" /
        f"DRWeibo_{window}min.jsonl",

        BASE_DIR /
        "DRWeibo" /
        f"drweibo_{window}_min.jsonl",
    ]

    for path in candidates:

        if path.exists():
            return path

    searched = "\n".join(
        str(path)
        for path in candidates
    )

    raise FileNotFoundError(
        f"Cannot find {window}-min DRWeibo file.\n"
        f"Searched:\n{searched}"
    )


# ============================================================
# 33. LOCKED SPLIT IDS
# ============================================================

def load_fixed_split_ids():

    split_dir = (
        SPLIT_DIR /
        "DRWeibo"
    )

    split_ids = {}

    for split_name in [
        "train",
        "val",
        "test",
    ]:

        path = (
            split_dir /
            f"{split_name}.jsonl"
        )

        samples = load_jsonl(
            path
        )

        ids = [
            str(sample["id"])
            for sample in samples
        ]

        if len(ids) != len(set(ids)):

            raise ValueError(
                f"Duplicate IDs found in {path}"
            )

        split_ids[
            split_name
        ] = ids

    all_ids = (
        split_ids["train"]
        +
        split_ids["val"]
        +
        split_ids["test"]
    )

    if len(all_ids) != len(set(all_ids)):

        raise ValueError(
            "Train/val/test ID overlap detected."
        )

    return split_ids


# ============================================================
# 34. APPLY LOCKED SPLIT TO EARLY WINDOW
# ============================================================

def build_window_splits(
    window_path,
    split_ids
):

    samples = load_jsonl(
        window_path
    )

    sample_map = {}

    for sample in samples:

        sample_id = str(
            sample["id"]
        )

        if sample_id in sample_map:

            raise ValueError(
                f"Duplicate ID in "
                f"{window_path}: "
                f"{sample_id}"
            )

        sample_map[
            sample_id
        ] = sample

    window_splits = {}

    for split_name in [
        "train",
        "val",
        "test",
    ]:

        wanted_ids = (
            split_ids[
                split_name
            ]
        )

        missing = [
            sample_id
            for sample_id in wanted_ids
            if sample_id not in sample_map
        ]

        if missing:

            raise ValueError(
                f"{window_path.name}: "
                f"{len(missing)} IDs missing from "
                f"{split_name}. Examples: "
                f"{missing[:10]}"
            )

        window_splits[
            split_name
        ] = [
            sample_map[
                sample_id
            ]
            for sample_id in wanted_ids
        ]

    return window_splits


# ============================================================
# 35. METRICS
# ============================================================

def calculate_metrics(
    labels,
    predictions,
    id2label
):

    label_ids = list(
        id2label.keys()
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    macro_f1 = f1_score(
        labels,
        predictions,
        labels=label_ids,
        average="macro",
        zero_division=0
    )

    class_f1 = f1_score(
        labels,
        predictions,
        labels=label_ids,
        average=None,
        zero_division=0
    )

    report = classification_report(
        labels,
        predictions,
        labels=label_ids,
        target_names=[
            id2label[i]
            for i in label_ids
        ],
        output_dict=True,
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "class_f1": class_f1,
        "report": report,
    }


# ============================================================
# 36. EVALUATION
# ============================================================

@torch.no_grad()
def evaluate(
    model,
    dataloader,
    device,
    criterion,
    id2label,
    use_amp
):

    model.eval()

    total_loss = 0.0

    all_labels = []
    all_predictions = []
    all_sample_ids = []

    for batch in tqdm(
        dataloader,
        desc="Evaluating",
        leave=False
    ):

        input_ids = batch["input_ids"].to(
            device,
            non_blocking=True
        )

        attention_mask = batch["attention_mask"].to(
            device,
            non_blocking=True
        )

        edge_index = batch["edge_index"].to(
            device,
            non_blocking=True
        )

        conversation_ids = batch["conversation_ids"].to(
            device,
            non_blocking=True
        )

        graph_stats = batch["graph_stats"].to(
            device,
            non_blocking=True
        )

        labels = batch["labels"].to(
            device,
            non_blocking=True
        )

        batch_size = labels.size(0)

        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=use_amp
        ):

            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                edge_index=edge_index,
                conversation_ids=conversation_ids,
                graph_stats=graph_stats,
                batch_size=batch_size
            )

            loss = criterion(
                logits,
                labels
            )

        total_loss += loss.item()

        predictions = torch.argmax(
            logits,
            dim=1
        )

        all_labels.extend(
            labels.cpu().tolist()
        )

        all_predictions.extend(
            predictions.cpu().tolist()
        )

        all_sample_ids.extend(
            batch["sample_ids"]
        )

    metrics = calculate_metrics(
        all_labels,
        all_predictions,
        id2label
    )

    return {
        "loss":
            total_loss /
            max(len(dataloader), 1),

        **metrics,

        "labels":
            all_labels,

        "predictions":
            all_predictions,

        "sample_ids":
            all_sample_ids,
    }


# ============================================================
# 37. OPTIMIZER STEP
# ============================================================

def optimizer_update(
    model,
    optimizer,
    scaler,
    use_amp
):

    if use_amp:

        scaler.unscale_(
            optimizer
        )

    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        GRADIENT_CLIP
    )

    if use_amp:

        scaler.step(
            optimizer
        )

        scaler.update()

    else:

        optimizer.step()

    optimizer.zero_grad(
        set_to_none=True
    )


# ============================================================
# 38. TRAIN ONE EPOCH
# ============================================================

def train_one_epoch(
    model,
    dataloader,
    optimizer,
    criterion,
    device,
    scaler,
    use_amp
):

    model.train()

    optimizer.zero_grad(
        set_to_none=True
    )

    total_loss = 0.0
    num_batches = len(dataloader)

    progress = tqdm(
        enumerate(
            dataloader,
            start=1
        ),
        total=num_batches,
        desc="Training"
    )

    for step, batch in progress:

        input_ids = batch["input_ids"].to(
            device,
            non_blocking=True
        )

        attention_mask = batch["attention_mask"].to(
            device,
            non_blocking=True
        )

        edge_index = batch["edge_index"].to(
            device,
            non_blocking=True
        )

        conversation_ids = batch["conversation_ids"].to(
            device,
            non_blocking=True
        )

        graph_stats = batch["graph_stats"].to(
            device,
            non_blocking=True
        )

        labels = batch["labels"].to(
            device,
            non_blocking=True
        )

        batch_size = labels.size(0)

        try:

            with torch.amp.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=use_amp
            ):

                logits = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    edge_index=edge_index,
                    conversation_ids=conversation_ids,
                    graph_stats=graph_stats,
                    batch_size=batch_size
                )

                raw_loss = criterion(
                    logits,
                    labels
                )

                loss = (
                    raw_loss /
                    GRAD_ACCUM_STEPS
                )

            if use_amp:

                scaler.scale(
                    loss
                ).backward()

            else:

                loss.backward()

            total_loss += raw_loss.item()

            should_update = (
                step % GRAD_ACCUM_STEPS == 0
                or
                step == num_batches
            )

            if should_update:

                optimizer_update(
                    model,
                    optimizer,
                    scaler,
                    use_amp
                )

            progress.set_postfix(
                loss=f"{raw_loss.item():.4f}",
                nodes=input_ids.size(0),
                edges=edge_index.size(1)
            )

        except torch.OutOfMemoryError:

            print("\nCUDA OOM")
            print("Nodes:", input_ids.size(0))
            print("Edges:", edge_index.size(1))

            print_cuda_memory()

            optimizer.zero_grad(
                set_to_none=True
            )

            gc.collect()

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            raise

    return (
        total_loss /
        max(num_batches, 1)
    )


# ============================================================
# 39. BUILD MODEL
# ============================================================

def build_model(
    variant,
    model_dir,
    num_classes
):

    if variant == "semantic":

        return SemanticOnlyClassifier(
            DATASET,
            model_dir,
            num_classes
        )

    if variant == "gat":

        return GATOnlyClassifier(
            DATASET,
            model_dir,
            num_classes
        )

    if variant == "ssee":

        return GATSSEEClassifier(
            DATASET,
            model_dir,
            num_classes
        )

    if variant in {
        "wo_sem",
        "wo_agcc",
        "wo_ase",
    }:

        return SSEEAblationClassifier(
            DATASET,
            model_dir,
            num_classes,
            ablation=variant
        )

    raise ValueError(
        f"Unknown model variant: {variant}"
    )


# ============================================================
# 40. CONTROLLED OPTIMIZER
# ============================================================

def build_optimizer(model):

    encoder_parameters = list(
        model.encoder.parameters()
    )

    encoder_ids = {
        id(parameter)
        for parameter in encoder_parameters
    }

    new_parameters = [
        parameter
        for parameter in model.parameters()
        if id(parameter)
        not in encoder_ids
    ]

    return torch.optim.AdamW(
        [
            {
                "params":
                    encoder_parameters,
                "lr":
                    ENCODER_LR
            },
            {
                "params":
                    new_parameters,
                "lr":
                    NEW_MODULE_LR
            },
        ],
        weight_decay=WEIGHT_DECAY
    )


# ============================================================
# 41. SAVE PREDICTIONS
# ============================================================

def save_predictions(
    sample_ids,
    labels,
    predictions,
    id2label,
    path
):

    rows = []

    for sample_id, y_true, y_pred in zip(
        sample_ids,
        labels,
        predictions
    ):

        rows.append(
            {
                "sample_id":
                    sample_id,
                "true_id":
                    y_true,
                "true_label":
                    id2label[y_true],
                "pred_id":
                    y_pred,
                "pred_label":
                    id2label[y_pred],
            }
        )

    pd.DataFrame(
        rows
    ).to_csv(
        path,
        index=False,
        encoding="utf-8-sig"
    )


# ============================================================
# 42. ONE MODEL × ONE WINDOW


# ============================================================
# 42. PHEME 9-FOLD LEAVE-ONE-EVENT-OUT MULTI-SEED


# ============================================================
# 42. DRWEIBO 10MIN SSEE PARAMETER SENSITIVITY


# ============================================================
# 42. CASE STUDY / INTERPRETABILITY
#     DRWeibo 10min | Full SSEE seed42
#     - loads the retained best_model.pt
#     - extracts SEM edge evidence, AGCC gates, ASE residual strength
#     - optionally compares with saved GAT predictions
#     - RTX 4090 auto-calibration
#     - resumable by deterministic chunks
# ============================================================

from pathlib import Path
import json
import math
import os


WINDOW_MIN = 10
CASE_SEED = 42

# Resolve the actual mounted project root robustly.
PROJECT_ROOT_CANDIDATES = [
    Path("/autodl-fs/data/processed_scsr"),
    Path("/root/autodl-fs/processed_scsr"),
]

PROJECT_ROOT = next(
    (p for p in PROJECT_ROOT_CANDIDATES if p.exists()),
    BASE_DIR
)

CASE_RESULT_DIR = (
    PROJECT_ROOT /
    "experiment_results" /
    "case_study_drweibo_10min_ssee_seed42"
)

CHUNK_DIR = CASE_RESULT_DIR / "chunks"

CHECKPOINT_CANDIDATES = [
    PROJECT_ROOT /
    "experiment_results" /
    "drweibo_ssee_ablation_multiseed_safe4090" /
    "ssee_seed42" /
    "10min" /
    "best_model.pt",

    PROJECT_ROOT /
    "experiment_results" /
    "drweibo_parameter_sensitivity_10min" /
    "default__default" /
    "best_model.pt",
]

GAT_PREDICTION_CANDIDATES = [
    PROJECT_ROOT /
    "experiment_results" /
    "drweibo_ssee_ablation_multiseed_safe4090" /
    "gat_seed42" /
    "10min" /
    "test_predictions.csv",

    PROJECT_ROOT /
    "experiment_results" /
    "drweibo_early_multiseed" /
    "gat_seed42" /
    "10min" /
    "test_predictions.csv",
]

# We only do inference/interpretability here, so large batches are useful.
# Calibration uses the longest conversations and falls back automatically.
BATCH_CANDIDATES = [128, 96, 64, 48, 32, 24, 16, 8]
NODE_CHUNK_CANDIDATES = [256, 192, 128, 96, 64]

TARGET_PEAK_LOW_GIB = 12.0
TARGET_PEAK_HIGH_GIB = 18.5

# Number of final paper-ready cases.
N_DISAGREEMENT_CASES = 3
N_SPARSE_CORRECT_CASES = 2
N_ERROR_CASES = 2

TOP_K_SEM_EDGES = 5
TOP_K_AGCC_NODES = 5
TOP_K_ASE_NODES = 5


def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass


def is_cuda_oom(exc):
    text = str(exc).lower()
    return (
        isinstance(exc, torch.cuda.OutOfMemoryError)
        or
        "cuda out of memory" in text
    )


def resolve_checkpoint():
    for p in CHECKPOINT_CANDIDATES:
        if p.exists():
            return p

    raise FileNotFoundError(
        "No SSEE seed42 10min best_model.pt found.\n"
        "Checked:\n" +
        "\n".join(str(p) for p in CHECKPOINT_CANDIDATES)
    )


def resolve_gat_predictions():
    for p in GAT_PREDICTION_CANDIDATES:
        if p.exists():
            return p
    return None


def load_split_ids(which):
    p = (
        PROJECT_ROOT /
        "splits" /
        "DRWeibo" /
        f"{which}.jsonl"
    )

    ids = []

    with p.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                ids.append(
                    str(json.loads(line)["id"])
                )

    return ids


def load_test_samples():
    p = (
        PROJECT_ROOT /
        "DRWeibo" /
        f"drweibo_{WINDOW_MIN}min.jsonl"
    )

    rows = load_jsonl(p)
    by_id = {
        str(x["id"]): x
        for x in rows
    }

    test_ids = load_split_ids("test")

    missing = [
        x for x in test_ids
        if x not in by_id
    ]

    if missing:
        raise RuntimeError(
            f"{len(missing)} test IDs are missing from {p.name}. "
            f"Examples: {missing[:5]}"
        )

    return [
        by_id[x]
        for x in test_ids
    ]


def build_raw_lookup(samples):
    return {
        str(x["id"]): x
        for x in samples
    }


def make_loader(samples, tokenizer, label2id, batch_size):
    collator = GraphConversationCollator(
        tokenizer,
        MAX_LENGTH
    )

    kwargs = dict(
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        collate_fn=collator,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=(NUM_WORKERS > 0),
    )

    if NUM_WORKERS > 0:
        kwargs["prefetch_factor"] = 6

    return DataLoader(
        ConversationGraphDataset(
            samples,
            label2id
        ),
        **kwargs
    )


def reset_peak():
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()


def peak_gib():
    if not torch.cuda.is_available():
        return None
    return (
        torch.cuda.max_memory_allocated()
        / 1024**3
    )


def prepare_batch_tensors(batch, device):
    return {
        "input_ids": batch["input_ids"].to(
            device,
            non_blocking=True
        ),
        "attention_mask": batch["attention_mask"].to(
            device,
            non_blocking=True
        ),
        "edge_index": batch["edge_index"].to(
            device,
            non_blocking=True
        ),
        "conversation_ids": batch["conversation_ids"].to(
            device,
            non_blocking=True
        ),
        "graph_stats": batch["graph_stats"].to(
            device,
            non_blocking=True
        ),
        "labels": batch["labels"].to(
            device,
            non_blocking=True
        ),
    }


@torch.no_grad()
def forward_with_explanations(
    model,
    batch,
    device,
    use_amp,
):
    """
    Reproduces GATSSEEClassifier.forward(), but keeps the intermediate
    quantities required for a paper case study:
      - SEM edge attention alpha
      - AGCC node gate gamma
      - ASE correction / residual magnitude
    """

    t = prepare_batch_tensors(
        batch,
        device
    )

    input_ids = t["input_ids"]
    attention_mask = t["attention_mask"]
    edge_index = t["edge_index"]
    conversation_ids = t["conversation_ids"]
    graph_stats = t["graph_stats"]
    labels = t["labels"]

    batch_size = int(
        labels.size(0)
    )

    with torch.amp.autocast(
        device_type="cuda",
        dtype=torch.float16,
        enabled=use_amp
    ):
        # ----------------------------------------------------
        # Text encoder + GAT
        # ----------------------------------------------------
        h_x = model.encode_nodes(
            input_ids,
            attention_mask
        )

        num_nodes = h_x.size(0)

        graph_edge_index = prepare_edge_index(
            edge_index,
            num_nodes,
            MAKE_BIDIRECTIONAL
        )

        h_g = F.elu(
            model.gat1(
                h_x,
                graph_edge_index
            )
        )

        h_g = F.elu(
            model.gat2(
                h_g,
                graph_edge_index
            )
        )

        # ----------------------------------------------------
        # SEM, explicitly retaining alpha
        # ----------------------------------------------------
        src = graph_edge_index[0]
        dst = graph_edge_index[1]

        q_i = model.sem.query_proj(
            h_g[dst]
        )

        k_j = model.sem.key_proj(
            h_g[src]
        )

        evidence_hidden = torch.tanh(
            q_i + k_j
        )

        evidence_scores = (
            evidence_hidden
            *
            model.sem.evidence_vector
        ).sum(dim=-1)

        sem_alpha = scalar_edge_softmax(
            evidence_scores,
            dst,
            num_nodes
        )

        # model is in eval() so dropout is identity, but reproduce
        # the actual module call for consistency.
        sem_alpha_used = model.sem.dropout(
            sem_alpha
        ).to(dtype=h_g.dtype)

        messages = (
            h_g[src]
            *
            sem_alpha_used.unsqueeze(-1)
        )

        h_m = torch.zeros(
            h_g.shape,
            dtype=h_g.dtype,
            device=h_g.device
        )

        h_m.index_add_(
            0,
            dst,
            messages
        )

        # ----------------------------------------------------
        # AGCC, explicitly retaining gamma
        # ----------------------------------------------------
        hidden_dim = h_m.size(-1)

        global_context = torch.zeros(
            (batch_size, hidden_dim),
            dtype=h_m.dtype,
            device=h_m.device
        )

        global_context.index_add_(
            0,
            conversation_ids,
            h_m
        )

        counts = (
            torch.bincount(
                conversation_ids,
                minlength=batch_size
            )
            .clamp(min=1)
            .unsqueeze(1)
            .to(
                dtype=h_m.dtype,
                device=h_m.device
            )
        )

        global_context = (
            global_context /
            counts
        )

        node_global = (
            global_context[
                conversation_ids
            ]
        )

        node_stats = (
            graph_stats[
                conversation_ids
            ]
            .to(
                dtype=h_m.dtype
            )
        )

        gate_input = torch.cat(
            [
                h_m,
                node_global,
                node_stats,
            ],
            dim=-1
        )

        gamma = torch.sigmoid(
            model.agcc.gate(
                gate_input
            )
        )

        h_c = (
            h_m +
            gamma *
            node_global
        )

        # ----------------------------------------------------
        # ASE, explicitly retaining correction
        # ----------------------------------------------------
        ase_input = torch.cat(
            [
                h_c,
                node_stats,
            ],
            dim=-1
        )

        correction = model.ase.mlp(
            ase_input
        )

        h_e = (
            h_g +
            correction
        )

        h_graph = model.conversation_mean_pool(
            h_e,
            conversation_ids,
            batch_size
        )

        logits = model.classifier(
            model.dropout(
                h_graph
            )
        )

        probs = torch.softmax(
            logits.float(),
            dim=-1
        )

    return {
        "logits": logits.float().detach().cpu(),
        "probs": probs.detach().cpu(),
        "labels": labels.detach().cpu(),
        "graph_edge_index":
            graph_edge_index.detach().cpu(),
        "sem_alpha":
            sem_alpha.float().detach().cpu(),
        "gamma":
            gamma.float().detach().cpu(),
        "correction":
            correction.float().detach().cpu(),
        "h_g":
            h_g.float().detach().cpu(),
        "conversation_ids":
            conversation_ids.detach().cpu(),
    }


def local_original_edges(sample):
    nodes = sample.get("nodes", [])
    node_to_idx = {
        str(n["node_id"]): i
        for i, n in enumerate(nodes)
    }

    result = set()

    for edge in sample.get("edges", []):
        if (
            isinstance(edge, (list, tuple))
            and len(edge) == 2
            and str(edge[0]) in node_to_idx
            and str(edge[1]) in node_to_idx
        ):
            result.add(
                (
                    node_to_idx[str(edge[0])],
                    node_to_idx[str(edge[1])],
                )
            )

    return result


def short_text(text, max_chars=120):
    text = " ".join(
        str(text).split()
    )

    if len(text) <= max_chars:
        return text

    return (
        text[:max_chars - 1]
        + "…"
    )


def edge_relation(
    local_src,
    local_dst,
    original_edges,
):
    if local_src == local_dst:
        return "self"

    if (
        local_src,
        local_dst
    ) in original_edges:
        return "original"

    if (
        local_dst,
        local_src
    ) in original_edges:
        return "reverse"

    return "augmented"


def summarize_one_conversation(
    sample,
    sample_index_in_batch,
    node_start,
    node_end,
    outputs,
    id2label,
):
    nodes = sample.get("nodes", [])

    true_id = int(
        outputs["labels"][
            sample_index_in_batch
        ].item()
    )

    probs = (
        outputs["probs"][
            sample_index_in_batch
        ]
    )

    pred_id = int(
        torch.argmax(
            probs
        ).item()
    )

    confidence = float(
        probs[pred_id].item()
    )

    original_edges = (
        local_original_edges(
            sample
        )
    )

    conv_ids = outputs[
        "conversation_ids"
    ]

    # --------------------------------------------------------
    # Node-level AGCC / ASE
    # --------------------------------------------------------
    node_global_indices = list(
        range(
            node_start,
            node_end
        )
    )

    gamma = outputs["gamma"]
    correction = outputs["correction"]
    h_g = outputs["h_g"]

    node_rows = []

    for local_idx, global_idx in enumerate(
        node_global_indices
    ):
        gamma_mean = float(
            gamma[
                global_idx
            ].mean().item()
        )

        corr_norm = float(
            torch.linalg.vector_norm(
                correction[
                    global_idx
                ]
            ).item()
        )

        base_norm = float(
            torch.linalg.vector_norm(
                h_g[
                    global_idx
                ]
            ).item()
        )

        residual_ratio = (
            corr_norm /
            max(base_norm, 1e-8)
        )

        node = nodes[local_idx]

        node_rows.append({
            "local_node_index":
                local_idx,
            "node_id":
                str(node.get("node_id")),
            "node_type":
                str(node.get("node_type", "")),
            "time_min":
                node.get("time_min"),
            "text":
                short_text(
                    node.get("text", "")
                ),
            "agcc_gamma_mean":
                gamma_mean,
            "ase_correction_norm":
                corr_norm,
            "ase_residual_ratio":
                residual_ratio,
        })

    top_agcc_nodes = sorted(
        node_rows,
        key=lambda x: x[
            "agcc_gamma_mean"
        ],
        reverse=True
    )[:TOP_K_AGCC_NODES]

    top_ase_nodes = sorted(
        node_rows,
        key=lambda x: x[
            "ase_residual_ratio"
        ],
        reverse=True
    )[:TOP_K_ASE_NODES]

    # --------------------------------------------------------
    # SEM edge-level evidence
    # --------------------------------------------------------
    ge = outputs[
        "graph_edge_index"
    ]

    alpha = outputs[
        "sem_alpha"
    ]

    edge_rows = []

    for edge_idx in range(
        ge.size(1)
    ):
        src_global = int(
            ge[0, edge_idx].item()
        )
        dst_global = int(
            ge[1, edge_idx].item()
        )

        # Both endpoints should be inside the same conversation
        # after graph augmentation.
        if not (
            node_start
            <= src_global
            < node_end
            and
            node_start
            <= dst_global
            < node_end
        ):
            continue

        local_src = (
            src_global -
            node_start
        )

        local_dst = (
            dst_global -
            node_start
        )

        relation = edge_relation(
            local_src,
            local_dst,
            original_edges
        )

        src_node = nodes[
            local_src
        ]
        dst_node = nodes[
            local_dst
        ]

        edge_rows.append({
            "src_local":
                local_src,
            "dst_local":
                local_dst,
            "src_node_id":
                str(src_node.get("node_id")),
            "dst_node_id":
                str(dst_node.get("node_id")),
            "relation":
                relation,
            "sem_alpha":
                float(
                    alpha[
                        edge_idx
                    ].item()
                ),
            "src_text":
                short_text(
                    src_node.get("text", "")
                ),
            "dst_text":
                short_text(
                    dst_node.get("text", "")
                ),
        })

    # Prefer non-self relationships for the human-readable case study.
    non_self = [
        x for x in edge_rows
        if x["relation"] != "self"
    ]

    ranked_edges = (
        non_self
        if non_self
        else edge_rows
    )

    top_sem_edges = sorted(
        ranked_edges,
        key=lambda x: x["sem_alpha"],
        reverse=True
    )[:TOP_K_SEM_EDGES]

    gs = sample.get(
        "graph_statistics",
        {}
    )

    return {
        "sample_id":
            str(sample["id"]),
        "true_id":
            true_id,
        "true_label":
            id2label[true_id],
        "pred_id":
            pred_id,
        "pred_label":
            id2label[pred_id],
        "confidence":
            confidence,
        "correct":
            bool(
                pred_id == true_id
            ),
        "num_nodes":
            len(nodes),
        "num_edges":
            len(sample.get("edges", [])),
        "max_depth":
            gs.get("max_depth"),
        "root_branches":
            gs.get("root_branches"),
        "leaf_ratio":
            gs.get("leaf_ratio"),
        "density":
            gs.get("density"),
        "source_text":
            short_text(
                nodes[0].get("text", "")
                if nodes
                else ""
            ),
        "top_sem_edges":
            top_sem_edges,
        "top_agcc_nodes":
            top_agcc_nodes,
        "top_ase_nodes":
            top_ase_nodes,
        "all_node_signals":
            node_rows,
    }


def calibrate_batch(
    model,
    samples,
    tokenizer,
    label2id,
    device,
    use_amp,
):
    global BATCH_SIZE
    global NODE_CHUNK_SIZE

    config_path = (
        CASE_RESULT_DIR /
        "inference_config.json"
    )

    if config_path.exists():
        with config_path.open(
            "r",
            encoding="utf-8"
        ) as f:
            cfg = json.load(f)

        BATCH_SIZE = int(
            cfg["batch_size"]
        )
        NODE_CHUNK_SIZE = int(
            cfg["node_chunk_size"]
        )

        print(
            f"RESUME inference config | "
            f"batch={BATCH_SIZE} | "
            f"chunk={NODE_CHUNK_SIZE}"
        )

        return cfg

    stress = sorted(
        samples,
        key=lambda x: len(
            x.get("nodes", [])
        ),
        reverse=True
    )[:256]

    trials = []

    print("\nRTX 4090 CASE-STUDY INFERENCE CALIBRATION")

    for chunk in NODE_CHUNK_CANDIDATES:
        for batch_size in BATCH_CANDIDATES:
            BATCH_SIZE = batch_size
            NODE_CHUNK_SIZE = chunk

            cleanup_cuda()

            try:
                loader = make_loader(
                    stress,
                    tokenizer,
                    label2id,
                    batch_size
                )

                batch = next(
                    iter(loader)
                )

                reset_peak()

                _ = forward_with_explanations(
                    model,
                    batch,
                    device,
                    use_amp
                )

                if torch.cuda.is_available():
                    torch.cuda.synchronize()

                peak = peak_gib()

                trials.append({
                    "batch_size":
                        batch_size,
                    "node_chunk_size":
                        chunk,
                    "peak_cuda_gib":
                        peak,
                    "status":
                        "PASS",
                })

                print(
                    f"PASS | batch={batch_size:<3d} "
                    f"chunk={chunk:<3d} "
                    f"peak={peak:.2f} GiB"
                )

                pd.DataFrame(
                    trials
                ).to_csv(
                    CASE_RESULT_DIR /
                    "batch_calibration.csv",
                    index=False,
                    encoding="utf-8-sig"
                )

                if (
                    peak is not None
                    and
                    TARGET_PEAK_LOW_GIB
                    <= peak
                    <= TARGET_PEAK_HIGH_GIB
                ):
                    cfg = {
                        "batch_size":
                            batch_size,
                        "node_chunk_size":
                            chunk,
                        "peak_cuda_gib":
                            peak,
                    }

                    with config_path.open(
                        "w",
                        encoding="utf-8"
                    ) as f:
                        json.dump(
                            cfg,
                            f,
                            indent=2
                        )

                    return cfg

            except Exception as exc:
                if is_cuda_oom(exc):
                    print(
                        f"OOM  | batch={batch_size:<3d} "
                        f"chunk={chunk:<3d}"
                    )

                    trials.append({
                        "batch_size":
                            batch_size,
                        "node_chunk_size":
                            chunk,
                        "peak_cuda_gib":
                            None,
                        "status":
                            "OOM",
                    })

                    pd.DataFrame(
                        trials
                    ).to_csv(
                        CASE_RESULT_DIR /
                        "batch_calibration.csv",
                        index=False,
                        encoding="utf-8-sig"
                    )

                else:
                    raise

            finally:
                try:
                    del batch
                except Exception:
                    pass
                try:
                    del loader
                except Exception:
                    pass
                cleanup_cuda()

    # If all passing configurations stay below 12 GiB, choose the
    # largest successful configuration rather than forcing more VRAM.
    pass_rows = [
        x for x in trials
        if x["status"] == "PASS"
    ]

    if not pass_rows:
        raise RuntimeError(
            "All case-study inference calibration profiles OOM."
        )

    best = pass_rows[0]

    cfg = {
        "batch_size":
            int(best["batch_size"]),
        "node_chunk_size":
            int(best["node_chunk_size"]),
        "peak_cuda_gib":
            best["peak_cuda_gib"],
    }

    with config_path.open(
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            cfg,
            f,
            indent=2
        )

    return cfg


def chunk_path(chunk_index):
    return (
        CHUNK_DIR /
        f"chunk_{chunk_index:04d}.jsonl"
    )


def save_chunk(path, records):
    tmp = path.with_suffix(
        path.suffix + ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8"
    ) as f:
        for record in records:
            f.write(
                json.dumps(
                    record,
                    ensure_ascii=False
                )
                + "\n"
            )
        f.flush()
        os.fsync(f.fileno())

    os.replace(
        tmp,
        path
    )


def load_all_chunk_records():
    records = []

    for p in sorted(
        CHUNK_DIR.glob(
            "chunk_*.jsonl"
        )
    ):
        with p.open(
            "r",
            encoding="utf-8"
        ) as f:
            for line in f:
                if line.strip():
                    records.append(
                        json.loads(line)
                    )

    return records


def run_resumable_inference(
    model,
    samples,
    tokenizer,
    label2id,
    id2label,
    device,
    use_amp,
):
    CHUNK_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    raw_lookup = (
        build_raw_lookup(
            samples
        )
    )

    total_chunks = math.ceil(
        len(samples) /
        BATCH_SIZE
    )

    for chunk_idx in range(
        total_chunks
    ):
        out_path = chunk_path(
            chunk_idx
        )

        if out_path.exists():
            print(
                f"SKIP completed chunk "
                f"{chunk_idx + 1}/{total_chunks}"
            )
            continue

        start = (
            chunk_idx *
            BATCH_SIZE
        )

        end = min(
            start + BATCH_SIZE,
            len(samples)
        )

        chunk_samples = (
            samples[start:end]
        )

        loader = make_loader(
            chunk_samples,
            tokenizer,
            label2id,
            len(chunk_samples)
        )

        batch = next(
            iter(loader)
        )

        print(
            f"Processing chunk "
            f"{chunk_idx + 1}/{total_chunks} "
            f"| conversations={len(chunk_samples)}"
        )

        reset_peak()

        outputs = (
            forward_with_explanations(
                model,
                batch,
                device,
                use_amp
            )
        )

        sample_ids = (
            batch["sample_ids"]
        )

        # Node offsets inside the collated flattened graph.
        conv_ids = outputs[
            "conversation_ids"
        ]

        records = []

        for conv_idx, sample_id in enumerate(
            sample_ids
        ):
            node_indices = torch.nonzero(
                conv_ids == conv_idx,
                as_tuple=False
            ).flatten()

            if node_indices.numel() == 0:
                raise RuntimeError(
                    f"No nodes for conversation {sample_id}"
                )

            node_start = int(
                node_indices.min().item()
            )

            node_end = int(
                node_indices.max().item()
            ) + 1

            sample = raw_lookup[
                str(sample_id)
            ]

            record = summarize_one_conversation(
                sample=sample,
                sample_index_in_batch=conv_idx,
                node_start=node_start,
                node_end=node_end,
                outputs=outputs,
                id2label=id2label,
            )

            records.append(
                record
            )

        save_chunk(
            out_path,
            records
        )

        p = peak_gib()

        if p is not None:
            print(
                f"Chunk peak CUDA: "
                f"{p:.2f} GiB"
            )

        del outputs
        del batch
        del loader
        cleanup_cuda()


def attach_gat_predictions(df):
    gat_path = (
        resolve_gat_predictions()
    )

    if gat_path is None:
        print(
            "\nNo saved GAT test_predictions.csv found. "
            "Case selection will proceed without GAT disagreement filtering."
        )

        df["gat_pred_label"] = None
        df["gat_correct"] = None
        df["ssee_correct_gat_wrong"] = False

        return df, None

    print(
        f"\nUsing saved GAT predictions:\n"
        f"  {gat_path}"
    )

    gat = pd.read_csv(
        gat_path,
        dtype={
            "sample_id": str,
            "pred_label": str,
            "true_label": str,
        }
    )

    gat = gat[
        [
            "sample_id",
            "pred_label",
            "true_label",
        ]
    ].rename(
        columns={
            "pred_label":
                "gat_pred_label",
            "true_label":
                "gat_true_label",
        }
    )

    # IMPORTANT: normalize label dtypes before comparison.
    # The previous version compared integer labels from the GAT CSV
    # with string labels produced by id2label, making every GAT
    # prediction appear wrong even when the values were identical.
    def _norm_label(v):
        if pd.isna(v):
            return None
        s = str(v).strip()
        if s.endswith(".0"):
            s = s[:-2]
        return s

    df["true_label"] = df["true_label"].map(_norm_label)
    df["pred_label"] = df["pred_label"].map(_norm_label)
    gat["gat_pred_label"] = gat["gat_pred_label"].map(_norm_label)
    gat["gat_true_label"] = gat["gat_true_label"].map(_norm_label)

    df = df.merge(
        gat,
        on="sample_id",
        how="left"
    )

    df["gat_correct"] = (
        df["gat_pred_label"].notna()
        &
        (
            df["gat_pred_label"]
            ==
            df["true_label"]
        )
    )

    df["ssee_correct_gat_wrong"] = (
        df["correct"]
        &
        (~df["gat_correct"].fillna(False))
    )

    return df, str(gat_path)


def select_cases(df):
    selected = []
    used = set()

    def add_rows(frame, category, n):
        nonlocal selected, used

        for _, row in frame.iterrows():
            sid = str(
                row["sample_id"]
            )

            if sid in used:
                continue

            item = row.to_dict()
            item["case_category"] = category

            selected.append(
                item
            )

            used.add(sid)

            if sum(
                1
                for x in selected
                if x["case_category"] == category
            ) >= n:
                break

    # --------------------------------------------------------
    # 1. Strongest evidence: SSEE correct while GAT was wrong.
    # Favor sparse graphs, then high confidence.
    # --------------------------------------------------------
    disagreement = df[
        (df["ssee_correct_gat_wrong"] == True)
        &
        (df["num_nodes"] >= 2)
        &
        (df["num_edges"] >= 1)
    ].copy()

    if len(disagreement) > 0:
        # Prefer genuinely sparse but non-trivial structures.
        disagreement["sparsity_distance"] = (
            disagreement["num_nodes"] - 3
        ).abs()

        disagreement = disagreement.sort_values(
            [
                "sparsity_distance",
                "confidence",
                "max_ase_residual_ratio",
            ],
            ascending=[
                True,
                False,
                False,
            ]
        )

        add_rows(
            disagreement,
            "SSEE-correct / GAT-wrong",
            N_DISAGREEMENT_CASES
        )

    # --------------------------------------------------------
    # 2. Very sparse but correctly verified.
    # --------------------------------------------------------
    correct = df[
        (df["correct"] == True)
        &
        (df["num_nodes"] >= 2)
        &
        (df["num_edges"] >= 1)
        &
        (df["num_nodes"] <= 6)
    ].copy()

    if len(correct) > 0:
        correct = correct.sort_values(
            [
                "num_nodes",
                "confidence",
                "max_ase_residual_ratio",
            ],
            ascending=[
                True,
                False,
                False,
            ]
        )

        add_rows(
            correct,
            "Sparse correct",
            N_SPARSE_CORRECT_CASES
        )

    # --------------------------------------------------------
    # 3. Failure cases for balanced discussion.
    # Rank high-confidence mistakes first.
    # --------------------------------------------------------
    wrong = df[
        (df["correct"] == False)
        &
        (df["num_nodes"] >= 2)
        &
        (df["num_edges"] >= 1)
    ].copy()

    if len(wrong) > 0:
        wrong = wrong.sort_values(
            [
                "confidence",
                "num_nodes",
            ],
            ascending=[
                False,
                True,
            ]
        )

        add_rows(
            wrong,
            "SSEE error",
            N_ERROR_CASES
        )

    return pd.DataFrame(
        selected
    )


def make_case_report(
    selected_df,
    detailed_by_id,
    checkpoint_path,
    gat_path,
):
    lines = []

    lines.append(
        "# DRWeibo 10min SSEE Case Study"
    )
    lines.append("")
    lines.append(
        f"- Checkpoint: `{checkpoint_path}`"
    )
    lines.append(
        f"- GAT predictions: `{gat_path}`"
        if gat_path
        else
        "- GAT predictions: not available"
    )
    lines.append(
        f"- Selected cases: {len(selected_df)}"
    )
    lines.append("")

    for idx, row in selected_df.reset_index(
        drop=True
    ).iterrows():
        sid = str(
            row["sample_id"]
        )

        detail = detailed_by_id[
            sid
        ]

        lines.append(
            f"## Case {idx + 1}: "
            f"{row['case_category']}"
        )
        lines.append("")

        lines.append(
            f"- Sample ID: `{sid}`"
        )
        lines.append(
            f"- True / SSEE: "
            f"{row['true_label']} / "
            f"{row['pred_label']}"
        )

        if (
            "gat_pred_label" in row
            and
            pd.notna(
                row["gat_pred_label"]
            )
        ):
            lines.append(
                f"- GAT prediction: "
                f"{row['gat_pred_label']}"
            )

        lines.append(
            f"- SSEE confidence: "
            f"{float(row['confidence']):.4f}"
        )
        lines.append(
            f"- Graph: "
            f"{int(row['num_nodes'])} nodes, "
            f"{int(row['num_edges'])} edges, "
            f"depth={row['max_depth']}"
        )
        lines.append(
            f"- Source: {row['source_text']}"
        )
        lines.append("")

        lines.append(
            "### Top SEM evidence edges"
        )
        lines.append("")

        for edge in detail[
            "top_sem_edges"
        ]:
            lines.append(
                f"- α={edge['sem_alpha']:.4f} "
                f"[{edge['relation']}] "
                f"`{edge['src_local']}→{edge['dst_local']}`: "
                f"{edge['src_text']} → {edge['dst_text']}"
            )

        lines.append("")
        lines.append(
            "### Highest AGCC compensation gates"
        )
        lines.append("")

        for node in detail[
            "top_agcc_nodes"
        ]:
            lines.append(
                f"- γ̄={node['agcc_gamma_mean']:.4f} "
                f"node {node['local_node_index']} "
                f"({node['node_type']}): "
                f"{node['text']}"
            )

        lines.append("")
        lines.append(
            "### Strongest ASE residual corrections"
        )
        lines.append("")

        for node in detail[
            "top_ase_nodes"
        ]:
            lines.append(
                f"- residual ratio="
                f"{node['ase_residual_ratio']:.4f} "
                f"node {node['local_node_index']} "
                f"({node['node_type']}): "
                f"{node['text']}"
            )

        lines.append("")

    report_path = (
        CASE_RESULT_DIR /
        "case_study_report.md"
    )

    report_path.write_text(
        "\n".join(lines),
        encoding="utf-8"
    )


def finalize_outputs(
    checkpoint_path,
):
    records = (
        load_all_chunk_records()
    )

    if not records:
        raise RuntimeError(
            "No chunk results found."
        )

    detailed_by_id = {
        str(x["sample_id"]): x
        for x in records
    }

    # Flat table for analysis / sorting.
    flat_rows = []

    for r in records:
        flat_rows.append({
            "sample_id":
                r["sample_id"],
            "true_label":
                r["true_label"],
            "pred_label":
                r["pred_label"],
            "confidence":
                r["confidence"],
            "correct":
                r["correct"],
            "num_nodes":
                r["num_nodes"],
            "num_edges":
                r["num_edges"],
            "max_depth":
                r["max_depth"],
            "root_branches":
                r["root_branches"],
            "leaf_ratio":
                r["leaf_ratio"],
            "density":
                r["density"],
            "source_text":
                r["source_text"],
            "mean_top_sem_alpha":
                (
                    float(np.mean([
                        x["sem_alpha"]
                        for x in r[
                            "top_sem_edges"
                        ]
                    ]))
                    if r["top_sem_edges"]
                    else np.nan
                ),
            "max_agcc_gamma_mean":
                (
                    max([
                        x["agcc_gamma_mean"]
                        for x in r[
                            "all_node_signals"
                        ]
                    ])
                    if r["all_node_signals"]
                    else np.nan
                ),
            "max_ase_residual_ratio":
                (
                    max([
                        x["ase_residual_ratio"]
                        for x in r[
                            "all_node_signals"
                        ]
                    ])
                    if r["all_node_signals"]
                    else np.nan
                ),
        })

    df = pd.DataFrame(
        flat_rows
    )

    df, gat_path = (
        attach_gat_predictions(
            df
        )
    )

    df.to_csv(
        CASE_RESULT_DIR /
        "all_test_interpretability.csv",
        index=False,
        encoding="utf-8-sig"
    )

    selected = select_cases(
        df
    )

    selected.to_csv(
        CASE_RESULT_DIR /
        "selected_case_studies.csv",
        index=False,
        encoding="utf-8-sig"
    )

    selected_details = []

    for _, row in selected.iterrows():
        sid = str(
            row["sample_id"]
        )

        detail = dict(
            detailed_by_id[sid]
        )

        detail["case_category"] = (
            row["case_category"]
        )

        if (
            "gat_pred_label" in row
            and
            pd.notna(
                row["gat_pred_label"]
            )
        ):
            detail["gat_pred_label"] = (
                row["gat_pred_label"]
            )

        selected_details.append(
            detail
        )

    with (
        CASE_RESULT_DIR /
        "selected_case_studies.json"
    ).open(
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            selected_details,
            f,
            ensure_ascii=False,
            indent=2
        )

    make_case_report(
        selected,
        detailed_by_id,
        checkpoint_path,
        gat_path,
    )

    summary = {
        "n_test":
            len(df),
        "n_correct":
            int(df["correct"].sum()),
        "n_wrong":
            int((~df["correct"]).sum()),
        "accuracy":
            float(df["correct"].mean()),
        "n_ssee_correct_gat_wrong":
            int(
                df[
                    "ssee_correct_gat_wrong"
                ].sum()
            ),
        "n_selected":
            len(selected),
        "checkpoint":
            str(checkpoint_path),
        "gat_predictions":
            gat_path,
    }

    with (
        CASE_RESULT_DIR /
        "case_study_summary.json"
    ).open(
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            summary,
            f,
            indent=2,
            ensure_ascii=False
        )

    print("\nSelected paper-ready cases:")
    if len(selected):
        print(
            selected[
                [
                    "case_category",
                    "sample_id",
                    "true_label",
                    "pred_label",
                    "confidence",
                    "num_nodes",
                    "num_edges",
                ]
            ].to_string(
                index=False
            )
        )

    print(
        f"\nSSEE-correct / GAT-wrong "
        f"cases available: "
        f"{summary['n_ssee_correct_gat_wrong']}"
    )


def main():
    global BATCH_SIZE
    global NODE_CHUNK_SIZE

    print("\n" + "=" * 96)
    print("DRWEIBO 10MIN SSEE CASE STUDY / INTERPRETABILITY")
    print("RTX 4090 AUTO-CALIBRATION + RESUMABLE CHUNKS")
    print("=" * 96)

    CASE_RESULT_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    CHUNK_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    set_seed(
        CASE_SEED
    )

    label2id, id2label = (
        get_label_mapping(
            "DRWeibo"
        )
    )

    tokenizer = load_tokenizer(
        "DRWeibo",
        DRWEIBO_MODEL_DIR
    )

    device = get_device()
    use_amp = (
        device.type == "cuda"
    )

    samples = load_test_samples()

    print(
        f"Test conversations: "
        f"{len(samples)}"
    )

    checkpoint_path = (
        resolve_checkpoint()
    )

    print(
        f"Checkpoint:\n"
        f"  {checkpoint_path}"
    )

    model = build_model(
        "ssee",
        DRWEIBO_MODEL_DIR,
        len(label2id)
    ).to(device)

    checkpoint = torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=False
    )

    model.load_state_dict(
        checkpoint[
            "model_state_dict"
        ]
    )

    model.eval()

    # Disable gradient checkpointing for inference.
    if hasattr(
        model.encoder,
        "gradient_checkpointing_disable"
    ):
        model.encoder.gradient_checkpointing_disable()

    cfg = calibrate_batch(
        model,
        samples,
        tokenizer,
        label2id,
        device,
        use_amp,
    )

    BATCH_SIZE = int(
        cfg["batch_size"]
    )

    NODE_CHUNK_SIZE = int(
        cfg["node_chunk_size"]
    )

    print(
        f"\nSelected case-study profile | "
        f"batch={BATCH_SIZE} | "
        f"node_chunk={NODE_CHUNK_SIZE} | "
        f"calibration_peak="
        f"{cfg.get('peak_cuda_gib')}"
    )

    # If batch config changed after earlier partial processing, refuse to
    # silently mix different chunk boundaries.
    manifest_path = (
        CASE_RESULT_DIR /
        "run_manifest.json"
    )

    manifest = {
        "dataset":
            "DRWeibo",
        "window_min":
            WINDOW_MIN,
        "seed":
            CASE_SEED,
        "batch_size":
            BATCH_SIZE,
        "node_chunk_size":
            NODE_CHUNK_SIZE,
        "checkpoint":
            str(checkpoint_path),
        "n_test":
            len(samples),
    }

    if manifest_path.exists():
        old = json.loads(
            manifest_path.read_text(
                encoding="utf-8"
            )
        )

        for key in [
            "batch_size",
            "node_chunk_size",
            "checkpoint",
            "n_test",
        ]:
            if str(old.get(key)) != str(
                manifest.get(key)
            ):
                raise RuntimeError(
                    "Existing partial case-study chunks were generated "
                    "under a different configuration. "
                    "Do not mix them. Move/delete the result directory "
                    "or restore the previous run_manifest.json config."
                )

    else:
        manifest_path.write_text(
            json.dumps(
                manifest,
                indent=2,
                ensure_ascii=False
            ),
            encoding="utf-8"
        )

    run_resumable_inference(
        model=model,
        samples=samples,
        tokenizer=tokenizer,
        label2id=label2id,
        id2label=id2label,
        device=device,
        use_amp=use_amp,
    )

    finalize_outputs(
        checkpoint_path
    )

    print("\n" + "=" * 96)
    print("CASE STUDY FINISHED")
    print("=" * 96)

    print(
        f"Results directory:\n"
        f"{CASE_RESULT_DIR}"
    )

    print("\nKey files:")
    for name in [
        "all_test_interpretability.csv",
        "selected_case_studies.csv",
        "selected_case_studies.json",
        "case_study_report.md",
        "case_study_summary.json",
        "batch_calibration.csv",
        "inference_config.json",
        "run_manifest.json",
    ]:
        print(
            CASE_RESULT_DIR /
            name
        )


if __name__ == "__main__":
    main()